[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/16_planificacion_clasica/notebooks/01_strips_y_estados.ipynb)

# Notebook 01 — STRIPS y estados

En este notebook implementamos la representación STRIPS para el dominio de **Blocks World** (mundo de bloques) con tres bloques: **A**, **B** y **C**.

Cubriremos:

1. Representación de estados como conjuntos de proposiciones.
2. Representación de acciones STRIPS (precondiciones, lista add, lista delete).
3. Las 18 acciones concretas del dominio.
4. Aplicación de acciones y ejecución de planes.
5. Generación del espacio de estados completo mediante BFS.
6. Visualización del grafo de estados.

---

## Blocks World y STRIPS

**Blocks World** es un dominio clásico de planificación. Tenemos bloques sobre una mesa y queremos reorganizarlos. Las reglas son:

- Solo se puede mover un bloque que tenga la parte superior libre (*clear*).
- Un bloque se puede colocar sobre la mesa o sobre otro bloque libre.
- La mesa tiene capacidad infinita.

**STRIPS** (Stanford Research Institute Problem Solver, 1971) describe acciones con tres componentes:

| Componente | Descripción |
|---|---|
| **Precondiciones** | Proposiciones que deben ser verdaderas para ejecutar la acción |
| **Lista Add** | Proposiciones que se vuelven verdaderas al ejecutar la acción |
| **Lista Delete** | Proposiciones que dejan de ser verdaderas al ejecutar la acción |

Un estado es un conjunto de proposiciones verdaderas. Todo lo que no aparece en el conjunto se considera falso (*suposición de mundo cerrado*).

---

## 1. Representación de estados

Representamos un estado como un `frozenset` de cadenas. Cada cadena es una proposición como `"On(A,Mesa)"` o `"Clear(B)"`. Usamos `frozenset` en lugar de `set` porque es inmutable y *hashable*, lo que nos permitirá usarlo como clave en diccionarios y como elemento de conjuntos (necesario para BFS).

También definimos una función `mostrar_estado()` que imprime las proposiciones de forma ordenada y legible.

In [ ]:
def mostrar_estado(estado, titulo="Estado"):
    """Pretty-print a STRIPS state.

    Parameters
    ----------
    estado : frozenset[str]
        Set of true propositions.
    titulo : str
        Header to display above the propositions.
    """
    print(f"\n{'=' * 40}")
    print(f"  {titulo}")
    print(f"{'=' * 40}")
    # Sort propositions alphabetically for consistent display
    for prop in sorted(estado):
        print(f"  - {prop}")
    print()


# ----- Estado inicial: los tres bloques sobre la mesa -----
s0 = frozenset({
    "On(A,Mesa)", "On(B,Mesa)", "On(C,Mesa)",
    "Clear(A)",   "Clear(B)",   "Clear(C)",
})

mostrar_estado(s0, "Estado inicial s0")

---

## 2. Representación de acciones

Usamos una `namedtuple` llamada `Action` con cuatro campos: `name`, `preconditions`, `add_list` y `delete_list`. Los tres últimos son `frozenset`s de proposiciones.

Definimos dos funciones:

- `is_applicable(action, state)`: verifica si todas las precondiciones de la acción están presentes en el estado.
- `apply_action(action, state)`: aplica la fórmula STRIPS $s' = (s - \text{Delete}) \cup \text{Add}$ y devuelve el nuevo estado.

In [ ]:
from collections import namedtuple

# Named tuple for a ground STRIPS action
Action = namedtuple("Action", ["name", "preconditions", "add_list", "delete_list"])


def is_applicable(action, state):
    """Check whether an action's preconditions are satisfied in the given state.

    Parameters
    ----------
    action : Action
        A ground STRIPS action.
    state : frozenset[str]
        Current state (set of true propositions).

    Returns
    -------
    bool
        True if all preconditions are in state.
    """
    # Every precondition must be present in the state
    return action.preconditions.issubset(state)


def apply_action(action, state):
    """Apply a STRIPS action to a state, returning the successor state.

    Implements the formula: s' = (s - delete_list) | add_list

    Parameters
    ----------
    action : Action
        A ground STRIPS action whose preconditions hold in state.
    state : frozenset[str]
        Current state.

    Returns
    -------
    frozenset[str]
        The successor state after applying the action.

    Raises
    ------
    ValueError
        If preconditions are not satisfied.
    """
    if not is_applicable(action, state):
        raise ValueError(
            f"Action {action.name} is not applicable: "
            f"missing {action.preconditions - state}"
        )
    # Remove delete_list, then add add_list
    return (state - action.delete_list) | action.add_list

---

## 3. Las 18 acciones concretas de Blocks World

Con tres bloques (A, B, C) tenemos tres esquemas de acción que generan 18 acciones concretas en total:

| Esquema | Descripción | Cantidad |
|---|---|:---:|
| `Mover(X, Y, Z)` | Mover X de bloque Y a bloque Z | 6 |
| `MoverAMesa(X, Y)` | Mover X de bloque Y a la mesa | 6 |
| `MoverDesdeMesa(X, Z)` | Mover X de la mesa a bloque Z | 6 |

A continuación definimos cada una de las 18 acciones con sus precondiciones, lista add y lista delete, exactamente como aparecen en las tablas de la clase.

In [ ]:
BLOCKS = ["A", "B", "C"]

# ---------- Generate all 18 ground actions ----------
all_actions = []

# --- Mover(X, Y, Z): move block X from block Y onto block Z ---
# X, Y, Z must all be different blocks (not Mesa)
for x in BLOCKS:
    for y in BLOCKS:
        for z in BLOCKS:
            if x != y and x != z and y != z:
                all_actions.append(Action(
                    name=f"Mover({x},{y},{z})",
                    preconditions=frozenset({f"On({x},{y})", f"Clear({x})", f"Clear({z})"}),
                    add_list=frozenset({f"On({x},{z})", f"Clear({y})"}),
                    delete_list=frozenset({f"On({x},{y})", f"Clear({z})"}),
                ))

# --- MoverAMesa(X, Y): move block X from block Y to the table ---
# X and Y must be different blocks
for x in BLOCKS:
    for y in BLOCKS:
        if x != y:
            all_actions.append(Action(
                name=f"MoverAMesa({x},{y})",
                preconditions=frozenset({f"On({x},{y})", f"Clear({x})"}),
                add_list=frozenset({f"On({x},Mesa)", f"Clear({y})"}),
                delete_list=frozenset({f"On({x},{y})"}),
            ))

# --- MoverDesdeMesa(X, Z): move block X from the table onto block Z ---
# X and Z must be different blocks
for x in BLOCKS:
    for z in BLOCKS:
        if x != z:
            all_actions.append(Action(
                name=f"MoverDesdeMesa({x},{z})",
                preconditions=frozenset({f"On({x},Mesa)", f"Clear({x})", f"Clear({z})"}),
                add_list=frozenset({f"On({x},{z})"}),
                delete_list=frozenset({f"On({x},Mesa)", f"Clear({z})"}),
            ))

# Verify we have exactly 18 actions
print(f"Total de acciones concretas: {len(all_actions)}")
print()
for i, a in enumerate(all_actions, 1):
    print(f"  {i:2d}. {a.name}")

---

## 4. Demo: acciones aplicables y aplicar una acción

Desde el estado inicial $s_0$ (los tres bloques en la mesa), enumeramos todas las acciones aplicables. Como todos los bloques están en la mesa y libres, solo las 6 acciones `MoverDesdeMesa` deberían ser aplicables.

Luego aplicamos `MoverDesdeMesa(B,C)` para obtener el estado $s_1$ y verificamos el resultado.

In [ ]:
# ---------- Find all applicable actions in s0 ----------
applicable = [a for a in all_actions if is_applicable(a, s0)]

print(f"Acciones aplicables en s0: {len(applicable)}\n")
for a in applicable:
    print(f"  - {a.name}")

# ---------- Apply MoverDesdeMesa(B,C) ----------
# Find the action by name
mover_bc = next(a for a in all_actions if a.name == "MoverDesdeMesa(B,C)")

s1 = apply_action(mover_bc, s0)

mostrar_estado(s0, "ANTES: s0")
print(f"  Acción aplicada: {mover_bc.name}")
print(f"    Delete: {sorted(mover_bc.delete_list)}")
print(f"    Add:    {sorted(mover_bc.add_list)}")
mostrar_estado(s1, "DESPUÉS: s1")

---

## 5. Ejecución manual de un plan paso a paso

Ejecutamos un plan de dos pasos para llegar a un estado donde B está sobre C y A está sobre B (torre A-B-C):

1. `MoverDesdeMesa(B, C)` — poner B sobre C.
2. `MoverDesdeMesa(A, B)` — poner A sobre B.

En cada paso mostramos el estado completo para verificar que las transiciones son correctas.

In [ ]:
# ---------- Manual plan execution ----------
plan = ["MoverDesdeMesa(B,C)", "MoverDesdeMesa(A,B)"]

# Build a lookup dictionary: action name -> Action
action_dict = {a.name: a for a in all_actions}

state = s0
mostrar_estado(state, "Paso 0 — Estado inicial")

for i, action_name in enumerate(plan, 1):
    action = action_dict[action_name]

    # Show which action we are about to apply
    print(f">>> Aplicar: {action.name}")
    print(f"    Precondiciones: {sorted(action.preconditions)}")
    print(f"    Satisfechas:    {is_applicable(action, state)}")

    # Apply the action
    state = apply_action(action, state)
    mostrar_estado(state, f"Paso {i} — Después de {action.name}")

# Verify we reached the goal: tower A on B on C on table
goal = frozenset({"On(A,B)", "On(B,C)", "On(C,Mesa)", "Clear(A)"})
print(f"Meta alcanzada: {goal.issubset(state)}")

---

## 6. Espacio de estados completo (BFS)

Generamos **todos** los estados alcanzables desde $s_0$ usando búsqueda en amplitud (BFS). En cada estado, probamos todas las acciones aplicables y registramos las transiciones.

Para Blocks World con 3 bloques, el espacio tiene exactamente **13 estados**:

| Bloques en mesa | Configuraciones |
|:---:|:---:|
| 3 | 1 |
| 2 | 6 |
| 1 | 6 |
| **Total** | **13** |

In [ ]:
from collections import deque


def generate_state_space(initial_state, actions):
    """Generate the full state space reachable from initial_state via BFS.

    Parameters
    ----------
    initial_state : frozenset[str]
        The starting state.
    actions : list[Action]
        All ground actions available.

    Returns
    -------
    states : list[frozenset[str]]
        All reachable states (in BFS order).
    edges : list[tuple[frozenset, str, frozenset]]
        Transitions as (source_state, action_name, target_state).
    """
    visited = {initial_state}   # Set of discovered states
    queue = deque([initial_state])
    states = [initial_state]
    edges = []

    while queue:
        current = queue.popleft()
        for action in actions:
            if is_applicable(action, current):
                successor = apply_action(action, current)
                # Record every transition (even to already-visited states)
                edges.append((current, action.name, successor))
                if successor not in visited:
                    visited.add(successor)
                    states.append(successor)
                    queue.append(successor)

    return states, edges


# ---------- Generate the full state space ----------
states, edges = generate_state_space(s0, all_actions)

print(f"Estados alcanzables: {len(states)}")
print(f"Transiciones totales: {len(edges)}")
print()

# Print all 13 states
for i, st in enumerate(states):
    mostrar_estado(st, f"Estado {i}")

---

## 7. Visualización del espacio de estados

Dibujamos el grafo de estados usando `matplotlib`. Cada nodo es un estado, representado por una etiqueta compacta que describe la configuración de los bloques. Las aristas conectan estados que difieren por una sola acción STRIPS.

Organizamos los nodos en tres niveles según el número de bloques en la mesa (3, 2 o 1) para facilitar la lectura.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np


def state_label(state):
    """Create a compact multi-line label describing the block configuration.

    Returns a string like 'A/B C' meaning A on B, C on table.
    A tower A-B-C becomes 'A/B/C'.
    """
    # Parse On(X, Y) propositions to build stacking info
    on = {}  # block -> what it sits on
    for prop in state:
        if prop.startswith("On("):
            # Parse "On(X,Y)" -> x, y
            inner = prop[3:-1]  # strip "On(" and ")"
            x, y = inner.split(",")
            on[x] = y

    # Build towers from bottom to top
    # Find blocks that sit on Mesa (tower bases)
    bases = [b for b in on if on[b] == "Mesa"]
    bases.sort()

    towers = []
    for base in bases:
        tower = [base]
        current = base
        # Follow the chain upward
        while True:
            # Find block sitting on current
            above = [b for b in on if on[b] == current]
            if above:
                current = above[0]
                tower.append(current)
            else:
                break
        # Display top-to-bottom: "A/B/C" means A on B on C on mesa
        towers.append("/".join(reversed(tower)))

    return "  ".join(towers)


def blocks_on_table(state):
    """Count how many blocks are directly on the table."""
    return sum(1 for p in state if p.startswith("On(") and p.endswith(",Mesa)"))


# ---------- Build adjacency (undirected edges) ----------
# Use state indices for graph building
state_to_idx = {s: i for i, s in enumerate(states)}

# Collect unique undirected edges
edge_set = set()
for src, action_name, dst in edges:
    i, j = state_to_idx[src], state_to_idx[dst]
    edge_set.add((min(i, j), max(i, j)))

# ---------- Layout: organize by number of blocks on table ----------
# Level 0 (top):    3 blocks on table -> 1 state
# Level 1 (middle): 2 blocks on table -> 6 states
# Level 2 (bottom): 1 block on table  -> 6 states

levels = {3: [], 2: [], 1: []}
for i, st in enumerate(states):
    n = blocks_on_table(st)
    levels[n].append(i)

# Sort within each level for consistent layout
for key in levels:
    levels[key].sort(key=lambda i: state_label(states[i]))

# Assign (x, y) positions
positions = {}
y_coords = {3: 2.0, 2: 1.0, 1: 0.0}

for level, idxs in levels.items():
    n = len(idxs)
    y = y_coords[level]
    # Center the nodes horizontally
    x_start = -(n - 1) / 2.0
    for k, idx in enumerate(idxs):
        positions[idx] = (x_start + k, y)

# ---------- Draw the graph ----------
fig, ax = plt.subplots(1, 1, figsize=(14, 8))

# Draw edges
for i, j in edge_set:
    xi, yi = positions[i]
    xj, yj = positions[j]
    ax.plot([xi, xj], [yi, yj], color="#888888", linewidth=0.8, zorder=1)

# Goal state: tower A on B on C
goal_state = frozenset({"On(A,B)", "On(B,C)", "On(C,Mesa)", "Clear(A)"})

# Draw nodes
for i, st in enumerate(states):
    x, y = positions[i]
    label = state_label(st)

    # Choose color based on special status
    if st == s0:
        color = "#c8e6c9"  # light green for initial state
        edge_color = "#2e7d32"
        linestyle = "--"
    elif st == goal_state:
        color = "#ffe0b2"  # light orange for goal
        edge_color = "#e65100"
        linestyle = "--"
    else:
        color = "#e3f2fd"  # light blue
        edge_color = "#1565c0"
        linestyle = "-"

    # Draw node circle
    circle = plt.Circle((x, y), 0.18, color=color, ec=edge_color,
                         linewidth=2, linestyle=linestyle, zorder=2)
    ax.add_patch(circle)

    # Draw label below the node
    ax.text(x, y - 0.28, label, ha="center", va="top",
            fontsize=7, fontfamily="monospace", fontweight="bold", zorder=3)

    # Draw state index inside the node
    ax.text(x, y, str(i), ha="center", va="center",
            fontsize=8, fontweight="bold", zorder=3)

# Labels for levels
ax.text(-3.8, 2.0, "3 en mesa", fontsize=10, va="center", fontstyle="italic", color="#555")
ax.text(-3.8, 1.0, "2 en mesa", fontsize=10, va="center", fontstyle="italic", color="#555")
ax.text(-3.8, 0.0, "1 en mesa", fontsize=10, va="center", fontstyle="italic", color="#555")

# Legend
legend_patches = [
    mpatches.Patch(facecolor="#c8e6c9", edgecolor="#2e7d32", linestyle="--", label="Estado inicial"),
    mpatches.Patch(facecolor="#ffe0b2", edgecolor="#e65100", linestyle="--", label="Estado meta (A/B/C)"),
    mpatches.Patch(facecolor="#e3f2fd", edgecolor="#1565c0", label="Otros estados"),
]
ax.legend(handles=legend_patches, loc="upper right", fontsize=9)

ax.set_xlim(-4.2, 4.2)
ax.set_ylim(-0.6, 2.6)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("Espacio de estados — Blocks World (A, B, C)",
             fontsize=14, fontweight="bold", pad=15)

plt.tight_layout()
plt.show()

print(f"\nNodos: {len(states)} | Aristas (sin dirección): {len(edge_set)}")